In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from huggingface_hub import notebook_login

notebook_login()
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [2]:
ds = load_dataset("openai/gsm8k", "main")
train_ds = ds["train"]
test_ds = ds["test"]

In [3]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [4]:
hidden_size = model.config.hidden_size
steering_vector = torch.randn(hidden_size, device=model.device, dtype=model.dtype)

In [5]:
layer_idx = 10
target_module = model.model.layers[layer_idx]

In [6]:
def make_hook(vec, coeff=1.0, token_position="last"):
    '''
    Wrapper class to make a hook function, called during the forward pass of the
    '''
    def hook(module, inputs, output):
        # Many HF modules return either:
        #   - a tensor
        #   - a tuple whose first element is the hidden states
        if isinstance(output, tuple):
            print(f'Output is a tuple of size {len(output)}')
            hidden = output[0]
        else:
            print(f'Module: {module}')
            print(f'Output is not a tuple, of dtype {type(output)}')
            hidden = output

        vec_cast = (coeff * vec).to(device=hidden.device, dtype=hidden.dtype)

        # hidden shape is usually [batch, seq, hidden_size]
        if token_position == "last":
            hidden = hidden.clone()
            hidden[:, -1, :] = hidden[:, -1, :] + vec_cast
        elif token_position == "all":
            hidden = hidden + vec_cast.view(1, 1, -1)
        else:
            raise ValueError("token_position must be 'last' or 'all'")

        if isinstance(output, tuple):
            return (hidden,) + output[1:]
        return hidden
    return hook

In [ ]:
handle = target_module.register_forward_hook(make_hook(steering_vector, coeff=1.0, token_position="last"))

In [ ]:

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model(**inputs)

Module: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=512, bias=False)
    (v_proj): Linear(in_features=2048, out_features=512, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
    (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
    (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
Output is not a tuple, of dtype <class 'torch.Tensor'>


In [16]:
inputs['input_ids'].shape

torch.Size([1, 6])

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
with torch.no_grad():


    handle.remove()
    out = model.generate(**inputs, max_new_tokens=20)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Output is not a tuple, of dtype <class 'torch.Tensor'>
Output is not a tuple, of dtype <class 'torch.Tensor'>
The capital of France is quint quint quint quint quint quintentlyently quint quint quint quint quint quint quint quint quint quintently quint
The capital of France is Paris. Paris is known for its rich history, cultural landmarks, and iconic landmarks such as the E
